# Pseudo-pair residual development probe

This unexecuted launcher changes only the deterministic translator parameterization. It reuses the observed development split, starts from a fresh initialization, trains for exactly 10 epochs, evaluates once at the endpoint, and keeps the scaled pilot blocked. Selected-slice results are exploratory and are not complete-volume or confirmatory evidence.

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import re
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")
REPOSITORY_URL = "https://github.com/GuillermoTafoya/MRIxFields.git"
REPO_DIR = Path("/content/FieldBridge-residual-probe")
EXPECTED_CODE_COMMIT = input("Residual-probe commit SHA: " ).strip()
MANIFEST_PATH = Path(input("External MRI manifest path: " ).strip()).expanduser()
PRIOR_SPLIT_PATH = Path(input("Prior Drive volume_splits.json path: " ).strip()).expanduser()
RUN_DIR = Path(input("New residual-probe run directory: " ).strip()).expanduser()
EXPECTED_SPLIT_SHA256 = "17f00411ab04331fa0380526b2d8f0cd0173e4ff6f8978f72c61053fa7385dbe"
CONFIG_NAME = "pseudo_pair_t2flair_residual_probe_10epoch.yaml"

if re.fullmatch(r"[0-9a-f]{40}", EXPECTED_CODE_COMMIT) is None:
    raise ValueError("Enter the exact 40-character residual-probe commit SHA.")
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError("The external MRI manifest path does not exist.")
if PRIOR_SPLIT_PATH.name != "volume_splits.json" or not PRIOR_SPLIT_PATH.is_file():
    raise FileNotFoundError("Provide the prior Drive volume_splits.json file.")
if RUN_DIR.exists():
    raise FileExistsError("Use a new run directory for fresh initialization.")
if REPO_DIR.exists():
    raise FileExistsError("Use a fresh Colab runtime or remove the old probe checkout.")

In [ ]:
subprocess.run(["git", "clone", REPOSITORY_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", "--detach", EXPECTED_CODE_COMMIT], cwd=REPO_DIR, check=True)
ACTUAL_CODE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if ACTUAL_CODE_COMMIT != EXPECTED_CODE_COMMIT:
    raise RuntimeError(f"Checked out {ACTUAL_CODE_COMMIT}, expected {EXPECTED_CODE_COMMIT}.")

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev,nifti,evaluation]"], cwd=REPO_DIR, check=True)
SOURCE_DIR = str(REPO_DIR / "src")
if SOURCE_DIR in sys.path:
    sys.path.remove(SOURCE_DIR)
sys.path.insert(0, SOURCE_DIR)
for module_name in tuple(sys.modules):
    if module_name == "fieldbridge" or module_name.startswith("fieldbridge."):
        del sys.modules[module_name]
importlib.invalidate_caches()
import fieldbridge as installed_fieldbridge

PACKAGE_FILE = Path(installed_fieldbridge.__file__).resolve()
if REPO_DIR not in PACKAGE_FILE.parents:
    raise RuntimeError(f"Same-kernel import did not resolve to the probe checkout: {PACKAGE_FILE}")
print({"code_commit": ACTUAL_CODE_COMMIT, "config": CONFIG_NAME, "package": str(PACKAGE_FILE)})

In [ ]:
RUNNER_PATH = REPO_DIR / "notebooks" / "pseudo_pair_residual_probe_runner.py"
runner_spec = importlib.util.spec_from_file_location("fieldbridge_residual_probe_runner", RUNNER_PATH)
if runner_spec is None or runner_spec.loader is None:
    raise RuntimeError("Unable to load the checked-in residual probe runner.")
runner = importlib.util.module_from_spec(runner_spec)
runner_spec.loader.exec_module(runner)

HANDOFF = runner.run_residual_probe(
    repo_dir=REPO_DIR,
    manifest_path=MANIFEST_PATH,
    prior_split_path=PRIOR_SPLIT_PATH,
    run_dir=RUN_DIR,
    code_commit=ACTUAL_CODE_COMMIT,
)

## Stop after handoff

Review the sanitized endpoint handoff. Do not launch the scaled pilot from this notebook. A failed restoration or conditioning sub-gate is valid development evidence.